# Phase 2 - Legal NER with Arch 1 - GPT-2 style decoder (word-level tokenizer)

Starts from the Phase-1 model (12-layer GPT-2 style decoder (111M), learned positions, word-level lowercased vocabulary.) and fine-tunes it to tag the 12 legal entity types with
**exact character offsets** (strict macro-F1).

Five experiments run back to back, spread over both T4s; each one produces its own submission zip:

| Experiment | What is trained |
|---|---|
| `full` | every weight, with layer-wise learning-rate decay |
| `lorma` | **LoRMA** (compulsory): multiplicative low-rank update `(alpha/r * BA + I) W x` on the attention projections |
| `lora` | LoRA: additive low-rank update on the same projections |
| `partial` | only the top layers + final norm |
| `kd` | knowledge distillation: the fine-tuned `full` model teaches a half-depth student |

How the pieces fit:
* **Tagging**: BIO tags over tokens, decoded back to character spans. Punctuation is split one character per piece so
  gold boundaries line up with token boundaries (checked below - an "oracle" decode of the gold tags shows the ceiling).
* **Long documents** (avg. ~31K characters): 512-token windows with 128-token overlap; each token keeps the prediction
  from the window where it has the most context on both sides.
* **Decoder-only models** see the whole window during fine-tuning (causal mask lifted), following Dukic & Snajder (2024)
  *Looking Right is Sometimes Right* and BehnamGhader et al. (2024) *LLM2Vec*.
* **Decoding**: plain argmax, or a Viterbi pass that only allows valid BIO sequences; the better one on the dev
  documents is used for the test predictions.

References: Hu et al. 2021 (LoRA); Bihany, Patel & Modi 2025 (LoRMA); Hinton et al. 2015 (distillation);
Sanh et al. 2019 (DistilBERT layer selection); Howard & Ruder 2018 and Clark et al. 2020 (layer-wise LR decay).

**Kaggle settings:** GPU T4 x2, Internet on, secret `HF` holding your Hugging Face token. Then Run All.

## 1. Setup

In [1]:
import os, sys, json, glob, shutil, time
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

WORK_DIR = "/kaggle/working/phase2_arch1"
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, WORK_DIR)

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GPUs: ['Tesla T4', 'Tesla T4']


## 2. Configuration
Everything you might want to change lives here.

In [2]:
%%writefile /kaggle/working/phase2_arch1/config.py
"""Every knob of this Phase-2 (legal NER) notebook."""
import os
import json

ARCH = "arch1"
WORK_DIR = f"/kaggle/working/phase2_{ARCH}"
DATA_DIR = os.path.join(WORK_DIR, "data")
CACHE_DIR = os.path.join(WORK_DIR, "phase1")        # Phase-1 files downloaded from Hugging Face land here
OUT_DIR = os.path.join(WORK_DIR, "runs")             # one folder per experiment

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
_P2 = os.path.join(DATA_DIR, "Phase_2_LNER")
FOLD_TEXT = [os.path.join(_P2, "Train", "fold1-text.json"), os.path.join(_P2, "Train", "fold2-text.json")]
FOLD_GOLD = [os.path.join(_P2, "Train", "fold1-gold.json"), os.path.join(_P2, "Train", "fold2-gold.json")]
TEST_TEXT = os.path.join(_P2, "Test", "fold3-text.json")

# ------------------------------------------------------------------ the Phase-1 model
# Looked up in this order: an attached Kaggle dataset whose path mentions "arch1", then these HF repos.
PHASE1_REPOS = ["D-240357/legal-gpt2-decoder"]
WEIGHT_FILES = ["model_state_dict.pt"]
CONFIG_FILES = ["config.json"]
TOKENIZER_FILES = ["custom_tokenizer.json"]
TOKENIZER_REPOS = PHASE1_REPOS
SPLIT_DIGITS = True            # split "Pradesh1" into "Pradesh" + "1" (see the alignment check in the notebook)
KAGGLE_HINT = "arch1"

NER_REPO = "D-240357/legal-ner-arch1"           # fine-tuned checkpoints, results and submission zips go here

# ------------------------------------------------------------------ experiments
# Each one is a separate submission. "kd" distils the fine-tuned KD_TEACHER into a half-depth student,
# so it waits for that run to finish.
METHODS = ["full", "lorma", "lora", "partial", "kd"]
KD_TEACHER = "full"

# ------------------------------------------------------------------ data
DEV_DOCS_PER_FOLD = 4          # 8 of the 70 training documents are held out to pick the best epoch / decoder
WINDOW = 512                   # tokens per window; documents average ~7K tokens
STRIDE = 384                   # 128-token overlap; every token is labelled from the window where it is most central
BIDIRECTIONAL = True            # lift the causal mask while fine-tuning (Dukic & Snajder 2024; LLM2Vec)
TRAIN_DOC_LIMIT = None         # set to e.g. 6 for a quick end-to-end smoke test

# ------------------------------------------------------------------ optimisation
EPOCHS = 10
PATIENCE = 3                   # stop after 3 epochs without a better dev F1
MICRO_BATCH = 8
GRAD_ACCUM = 2
EVAL_BATCH = 16
WARMUP_FRAC = 0.1
WEIGHT_DECAY = 0.01
HEAD_DROPOUT = 0.1
HEAD_LR = 1e-3

FULL_LR = 1e-4                 # top layer; each layer below gets LLRD times the one above it
LLRD = 0.85
PARTIAL_LR = 2e-4
PARTIAL_TOP_LAYERS = 4
ADAPTER_LR = 5e-4
ADAPTER_RANK = 8
LORA_ALPHA = 16
LORMA_ALPHA = 8
ADAPTER_DROPOUT = 0.05
ADAPTER_TARGETS = ["W_qkv", "W_o"]       # attention projections, as in the LoRA and LoRMA papers
CLASS_WEIGHTING = "none"       # or "sqrt_inv" to up-weight rare tags

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5                 # weight of the gold-label loss; the rest goes to matching the teacher
STUDENT_LAYERS = [0, 2, 4, 6, 8, 10]

SEED = 13

_override = os.environ.get("P2_CFG_OVERRIDE")
if _override:
    globals().update(json.loads(_override))

Writing /kaggle/working/phase2_arch1/config.py


## 3. Code
The Phase-1 tokenizer and model (unchanged, so the trained weights load), plus the NER pieces.

In [3]:
%%writefile /kaggle/working/phase2_arch1/tokenizer.py
"""Arch 1's word-level tokenizer (exactly as in Phase 1) plus an offset-aware encoder for NER."""
import re
import json
from collections import Counter


class CustomTokenizer:
    def __init__(self, vocab_size=32000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
        self.word2id = {w: i for i, w in enumerate(self.special_tokens)}
        self.id2word = {i: w for i, w in enumerate(self.special_tokens)}
        self.pad_id = 0
        self.unk_id = 1
        self.bos_id = 2
        self.eos_id = 3
        self.counter = Counter()

    def _tokenize_text(self, text):
        return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)

    def encode(self, text, add_special_tokens=False):
        ids = [self.word2id.get(tok, self.unk_id) for tok in self._tokenize_text(text.lower())]
        if add_special_tokens:
            ids = [self.bos_id] + ids + [self.eos_id]
        return ids

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.word2id = data["word2id"]
        self.id2word = {int(k): v for k, v in data["id2word"].items()}
        self.vocab_size = len(self.word2id)


# Phase 1 used \w+ which glues letters and digits ("Pradesh1", "ENT08"). Splitting them lifts the share of
# gold spans that start and end on a token boundary; both halves are ordinary vocabulary words.
NER_WORD_RE = re.compile(r"[^\W\d_]+|\d+|_+|[^\w\s]", re.UNICODE)


class NerTokenizer(CustomTokenizer):
    split_digits = True

    def encode_with_offsets(self, text):
        splitter = NER_WORD_RE if self.split_digits else re.compile(r"\w+|[^\w\s]", re.UNICODE)
        ids, starts, ends = [], [], []
        for m in splitter.finditer(text):
            ids.append(self.word2id.get(m.group().lower(), self.unk_id))
            starts.append(m.start())
            ends.append(m.end())
        return ids, starts, ends

Writing /kaggle/working/phase2_arch1/tokenizer.py


In [4]:
%%writefile /kaggle/working/phase2_arch1/phase1_model.py
"""
Arch 1 (GPT-2 style decoder) exactly as trained in Phase 1, so the saved weights load unchanged.
Phase 2 adds one switch: `causal`, so attention can look right as well as left while fine-tuning for NER.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout, max_len):
        super().__init__()
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.W_qkv = nn.Linear(d_model, 3 * d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.resid_drop = nn.Dropout(dropout)
        self.dropout_p = dropout
        self.causal = True

    def forward(self, x, key_mask=None):
        B, T, D = x.shape
        qkv = self.W_qkv(x).reshape(B, T, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        p = self.dropout_p if self.training else 0.0
        if key_mask is not None:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask, dropout_p=p)
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal, dropout_p=p)
        out = out.permute(0, 2, 1, 3).reshape(B, T, D)
        return self.resid_drop(self.W_o(out))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.up = nn.Linear(d_model, d_ff)
        self.down = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.down(F.gelu(self.up(x))))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, dropout, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, key_mask=None):
        x = x + self.attn(self.ln1(x), key_mask)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT2LM(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, d_ff, dropout, max_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_drop = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_head, d_ff, dropout, max_len)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_emb.weight

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.emb_drop(self.token_emb(x) + self.pos_emb(positions))
        for layer in self.layers:
            h = layer(h)
        return self.head(self.ln_f(h))

Writing /kaggle/working/phase2_arch1/phase1_model.py


In [5]:
%%writefile /kaggle/working/phase2_arch1/hub_files.py
"""Finds Phase-1 files: an attached Kaggle dataset first, then a local cache, then Hugging Face."""
import os
import re

from huggingface_hub import hf_hub_download

KAGGLE_INPUT = "/kaggle/input"


def _squash(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())


def find_in_kaggle_input(filename, hint):
    """Only accepts a file whose path mentions the architecture (e.g. 'arch3', 'arch-3', 'Arch_3'),
    because arch 2, 3 and 4 all call their weights model_final.pt."""
    if not os.path.isdir(KAGGLE_INPUT):
        return None
    hits = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        if filename in files:
            path = os.path.join(root, filename)
            if _squash(hint) in _squash(path):
                hits.append(path)
    return sorted(hits)[0] if hits else None


def fetch_from_hub(repo, filename, local_dir):
    try:
        return hf_hub_download(repo, filename, local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def locate(filenames, repos, hint, local_dir):
    """Names are tried in order of preference. Returns (path, where it came from)."""
    for name in filenames:
        path = find_in_kaggle_input(os.path.basename(name), hint)
        if path:
            return path, "kaggle input"
    for name in filenames:
        for repo in repos:
            cached = os.path.join(local_dir, _squash(repo), name)
            if os.path.exists(cached):
                return cached, f"cache of hf://{repo}"
            path = fetch_from_hub(repo, name, os.path.join(local_dir, _squash(repo)))
            if path:
                return path, f"hf://{repo}"
    raise FileNotFoundError(f"None of {filenames} found under /kaggle/input (path must mention '{hint}') or in "
                            f"{repos}. Attach the Phase-1 weights as a Kaggle dataset, or check HF_TOKEN and the repo names.")

Writing /kaggle/working/phase2_arch1/hub_files.py


In [6]:
%%writefile /kaggle/working/phase2_arch1/backbone.py
"""Loads the Phase-1 arch 1 weights and exposes them as a token encoder for the NER tagger."""
import json
import os

import torch
import torch.nn as nn

import config as C
from hub_files import locate, fetch_from_hub  # noqa: F401  (fetch_from_hub is used by run_ner)
from phase1_model import GPT2LM
from tokenizer import NerTokenizer


def _find(names, repos=None):
    return locate(names, repos or C.PHASE1_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)


def load_tokenizer():
    path, _ = _find(C.TOKENIZER_FILES, C.TOKENIZER_REPOS)
    tok = NerTokenizer()
    tok.load(path)
    tok.split_digits = C.SPLIT_DIGITS
    return tok


class Backbone(nn.Module):
    def __init__(self, gpt, keep_layers, bidirectional):
        super().__init__()
        self.d_model = gpt.token_emb.embedding_dim
        self.tok_emb = gpt.token_emb
        self.pos_emb = gpt.pos_emb
        self.emb_drop = gpt.emb_drop
        blocks = gpt.layers if keep_layers is None else [gpt.layers[i] for i in keep_layers]
        self.layers = nn.ModuleList(blocks)
        self.final_norm = gpt.ln_f
        self.bidirectional = bidirectional
        for blk in self.layers:
            blk.attn.causal = not bidirectional

    def forward(self, ids, mask):
        T = ids.size(1)
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        h = self.emb_drop(self.tok_emb(ids) + self.pos_emb(pos))
        key_mask = mask[:, None, None, :] if (self.bidirectional and not bool(mask.all())) else None
        for blk in self.layers:
            h = blk(h, key_mask)
        return self.final_norm(h)


def build_backbone(keep_layers=None, bidirectional=True):
    cfg_path, _ = _find(C.CONFIG_FILES)
    weights_path, source = _find(C.WEIGHT_FILES)
    with open(cfg_path) as f:
        cfg = json.load(f)
    gpt = GPT2LM(cfg["vocab_size"], cfg["d_model"], cfg["n_head"], cfg["n_layer"], cfg["d_ff"],
                 cfg["dropout"], cfg["max_len"])
    state = torch.load(weights_path, map_location="cpu")
    state = {k.replace("_orig_mod.", ""): v.float() for k, v in state.items()}
    gpt.load_state_dict(state)
    if os.environ.get("P2_VERBOSE", "1") == "1":
        print(f"Phase-1 arch 1 weights: {weights_path} ({source})", flush=True)
    return Backbone(gpt, keep_layers, bidirectional)


def student_layers():
    return list(C.STUDENT_LAYERS)

Writing /kaggle/working/phase2_arch1/backbone.py


In [7]:
%%writefile /kaggle/working/phase2_arch1/ner_core.py
"""
Phase 2 (legal NER) machinery shared by all four architectures:
data loading, BIO tagging, sliding windows, LoRA / LoRMA adapters, the tagging model,
training, span decoding and the strict (exact offsets + label) metric.
"""
import os
import json
import math
import random
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
TAGS = ["O"] + [f"{p}-{lab}" for lab in LABELS for p in ("B", "I")]
TAG_ID = {t: i for i, t in enumerate(TAGS)}
N_TAGS = len(TAGS)
IGNORE = -100


# --------------------------------------------------------------------------------------------- data
def load_docs(text_path, gold_path=None):
    with open(text_path, encoding="utf-8") as f:
        texts = json.load(f)
    gold = {}
    if gold_path:
        with open(gold_path, encoding="utf-8") as f:
            gold = json.load(f)
    docs = []
    for doc_id, text in texts.items():
        ents = [(int(e[0]), int(e[1]), e[2]) for e in gold.get(doc_id, [])]
        docs.append({"id": str(doc_id), "text": text, "ents": sorted(ents)})
    return docs


def split_dev(fold_docs, n_dev_per_fold, seed):
    """Hold out a few whole documents from every fold, so the dev set looks like the test set."""
    rng = random.Random(seed)
    train, dev = [], []
    for docs in fold_docs:
        docs = sorted(docs, key=lambda d: d["id"])
        rng.shuffle(docs)
        dev += docs[:n_dev_per_fold]
        train += docs[n_dev_per_fold:]
    return train, dev


# ------------------------------------------------------------------------------------ BIO encoding
def encode_doc(doc, tokenizer):
    """
    Tokenize with character offsets. starts/ends are the *content* span of every token (leading or
    trailing whitespace removed); pure-whitespace tokens get an empty span and never carry a label.
    """
    ids, starts, ends = tokenizer.encode_with_offsets(doc["text"])
    ids, starts, ends = np.asarray(ids, np.int64), np.asarray(starts, np.int64), np.asarray(ends, np.int64)
    real = ends > starts
    tags = np.where(real, TAG_ID["O"], IGNORE).astype(np.int64)

    for s, e, lab in doc["ents"]:
        hit = np.nonzero(real & (starts < e) & (ends > s))[0]
        if len(hit) == 0:
            continue
        tags[hit[0]] = TAG_ID[f"B-{lab}"]
        tags[hit[1:]] = TAG_ID[f"I-{lab}"]
    return {"id": doc["id"], "text": doc["text"], "ents": doc["ents"],
            "ids": ids, "starts": starts, "ends": ends, "tags": tags}


def make_windows(n, size, stride):
    if n <= size:
        return [(0, n)]
    wins = [(s, s + size) for s in range(0, n - size + 1, stride)]
    if wins[-1][1] < n:
        wins.append((n - size, n))
    return wins


def owner_of_tokens(windows, n):
    """For each token, the window in which it sits furthest from an edge (most context on both sides)."""
    best = np.full(n, -1, dtype=np.int64)
    best_margin = np.full(n, -1, dtype=np.int64)
    for k, (ws, we) in enumerate(windows):
        pos = np.arange(ws, we)
        margin = np.minimum(pos - ws, we - 1 - pos)
        better = margin > best_margin[ws:we]
        best[ws:we][better] = k
        best_margin[ws:we][better] = margin[better]
    return best


def window_items(enc_docs, size, stride):
    items = []
    for d, doc in enumerate(enc_docs):
        for ws, we in make_windows(len(doc["ids"]), size, stride):
            items.append((d, ws, we))
    return items


def collate(enc_docs, items, pad_id):
    T = max(we - ws for _, ws, we in items)
    B = len(items)
    ids = torch.full((B, T), pad_id, dtype=torch.long)
    tags = torch.full((B, T), IGNORE, dtype=torch.long)
    mask = torch.zeros((B, T), dtype=torch.bool)
    for r, (d, ws, we) in enumerate(items):
        doc = enc_docs[d]
        L = we - ws
        ids[r, :L] = torch.from_numpy(doc["ids"][ws:we])
        tags[r, :L] = torch.from_numpy(doc["tags"][ws:we])
        mask[r, :L] = True
    return ids, tags, mask


# ----------------------------------------------------------------------------------- adapters
class LoRALinear(nn.Module):
    """Hu et al. 2021: W x + (alpha / r) * B A x, with B = 0 at the start."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    """
    Bihany, Patel & Modi 2025 (LoRMA+): h = (alpha/r * B A + I) W0 x.
    Computed as W0x first, then A, then B, which costs the same as LoRA. B = 0 makes it the identity at the start.
    """

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        d = base.out_features
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + F.linear(F.linear(self.drop(h), self.A), self.B) * self.scale
        if self.base.bias is not None:
            h = h + self.base.bias
        return h


def inject_adapters(root, target_names, kind, r, alpha, dropout):
    wrapper = {"lora": LoRALinear, "lorma": LoRMALinear}[kind]
    swapped = 0
    for parent in list(root.modules()):
        for name, child in list(parent.named_children()):
            if name in target_names and isinstance(child, nn.Linear):
                setattr(parent, name, wrapper(child, r, alpha, dropout))
                swapped += 1
    return swapped


# --------------------------------------------------------------------------------------- model
class Tagger(nn.Module):
    def __init__(self, backbone, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(backbone.d_model, N_TAGS)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, ids, mask):
        h = self.backbone(ids, mask)
        return self.head(self.drop(h.float()))


def configure_method(model, method, cfg):
    """Freeze / unfreeze / add adapters according to the experiment. Returns a short description."""
    bb = model.backbone
    if method in ("full", "kd"):
        for p in model.parameters():
            p.requires_grad = True
        return "all weights trainable"

    for p in bb.parameters():
        p.requires_grad = False

    if method == "partial":
        top = list(bb.layers)[-cfg.PARTIAL_TOP_LAYERS:]
        for layer in top:
            for p in layer.parameters():
                p.requires_grad = True
        for p in bb.final_norm.parameters():
            p.requires_grad = True
        return f"top {cfg.PARTIAL_TOP_LAYERS} layers + final norm trainable"

    if method in ("lora", "lorma"):
        n = inject_adapters(bb.layers, cfg.ADAPTER_TARGETS, method, cfg.ADAPTER_RANK,
                            cfg.LORA_ALPHA if method == "lora" else cfg.LORMA_ALPHA, cfg.ADAPTER_DROPOUT)
        return f"{n} linear layers wrapped with {method.upper()} (r={cfg.ADAPTER_RANK})"

    raise ValueError(f"unknown method {method}")


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train


def trainable_state(model):
    """Only what training changed: adapters + head for PEFT runs, everything for full / KD."""
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().half() if v.is_floating_point() else v for k, v in model.state_dict().items() if k in names}


# ----------------------------------------------------------------------------------- optimizer
def layer_depth(name, n_layers):
    if ".layers." in name:
        return int(name.split(".layers.")[1].split(".")[0]) + 1
    if "emb" in name and name.startswith("backbone."):
        return 0
    return n_layers + 1                      # final norm and the head


def build_optimizer(model, method, cfg):
    n_layers = len(model.backbone.layers)
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("head."):
            lr = cfg.HEAD_LR
        elif method in ("full", "kd"):
            # layer-wise LR decay (ULMFiT / ELECTRA): lower layers move less
            lr = cfg.FULL_LR * cfg.LLRD ** (n_layers + 1 - layer_depth(name, n_layers))
        elif method == "partial":
            lr = cfg.PARTIAL_LR
        else:
            lr = cfg.ADAPTER_LR
        wd = 0.0 if p.ndim < 2 else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)


# ------------------------------------------------------------------------------------ decoding
def _allowed():
    allow = np.ones((N_TAGS, N_TAGS), dtype=bool)          # allow[prev, cur]
    for cur, tag in enumerate(TAGS):
        if tag.startswith("I-"):
            lab = tag[2:]
            allow[:, cur] = False
            allow[TAG_ID[f"B-{lab}"], cur] = True
            allow[TAG_ID[f"I-{lab}"], cur] = True
    start_ok = np.array([not t.startswith("I-") for t in TAGS])
    return allow, start_ok


ALLOW, START_OK = _allowed()


def viterbi(logp):
    """Best tag path that is a valid BIO sequence (no I-X without a preceding B-X / I-X)."""
    T = len(logp)
    if T == 0:
        return np.zeros(0, dtype=np.int64)
    trans = np.where(ALLOW, 0.0, -1e9)
    score = np.where(START_OK, logp[0], -1e9)
    back = np.zeros((T, N_TAGS), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = np.zeros(T, dtype=np.int64)
    path[-1] = score.argmax()
    for t in range(T - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path


def greedy(logp):
    return logp.argmax(-1)


def tags_to_spans(tag_ids, starts, ends):
    """Turn a tag sequence over non-whitespace tokens into (start, end, label) character spans."""
    spans, cur = [], None
    for t, s, e in zip(tag_ids, starts, ends):
        tag = TAGS[t]
        if tag == "O":
            if cur:
                spans.append(tuple(cur))
            cur = None
            continue
        kind, lab = tag.split("-", 1)
        if kind == "B" or cur is None or cur[2] != lab:     # an orphan I- simply opens a new entity
            if cur:
                spans.append(tuple(cur))
            cur = [int(s), int(e), lab]
        else:
            cur[1] = int(e)
    if cur:
        spans.append(tuple(cur))
    return spans


# -------------------------------------------------------------------------------------- metric
def strict_scores(pred, gold):
    """pred / gold: {doc_id: iterable of (start, end, label)}. Exact span AND label must match."""
    tp, n_pred, n_gold = ({lab: 0 for lab in LABELS} for _ in range(3))
    for doc_id in gold:
        g, p = set(map(tuple, gold[doc_id])), set(map(tuple, pred.get(doc_id, [])))
        for s, e, lab in g:
            n_gold[lab] += 1
        for s, e, lab in p:
            n_pred[lab] += 1
            if (s, e, lab) in g:
                tp[lab] += 1
    per = {}
    for lab in LABELS:
        if n_gold[lab] == 0 and n_pred[lab] == 0:
            continue
        p = tp[lab] / n_pred[lab] if n_pred[lab] else 0.0
        r = tp[lab] / n_gold[lab] if n_gold[lab] else 0.0
        per[lab] = {"p": p, "r": r, "f1": 2 * p * r / (p + r) if p + r else 0.0, "gold": n_gold[lab]}
    macro = {k: float(np.mean([v[k] for v in per.values()])) if per else 0.0 for k in ("p", "r", "f1")}
    return macro, per


def oracle_spans(enc_doc):
    """Decode the gold tags themselves: shows the best score this tokenization allows."""
    real = enc_doc["ends"] > enc_doc["starts"]
    return tags_to_spans(enc_doc["tags"][real], enc_doc["starts"][real], enc_doc["ends"][real])


# ----------------------------------------------------------------------------------- inference
@torch.no_grad()
def doc_log_probs(model, enc_docs, cfg, device, pad_id, batch_size):
    """Per-token log-probabilities for whole documents, stitched from overlapping windows."""
    model.eval()
    out = []
    for doc in enc_docs:
        n = len(doc["ids"])
        wins = make_windows(n, cfg.WINDOW, cfg.STRIDE)
        owner = owner_of_tokens(wins, n)
        lp = np.zeros((n, N_TAGS), dtype=np.float32)
        items = [(0, ws, we) for ws, we in wins]
        for b in range(0, len(items), batch_size):
            chunk = items[b:b + batch_size]
            ids, _, mask = collate([doc], chunk, pad_id)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(ids.to(device), mask.to(device))
            logp = F.log_softmax(logits.float(), -1).cpu().numpy()
            for r, (_, ws, we) in enumerate(chunk):
                k = b + r
                own = np.nonzero(owner[ws:we] == k)[0]
                lp[ws + own] = logp[r, own]
        out.append(lp)
    return out


def decode_docs(enc_docs, log_probs, decoder):
    fn = viterbi if decoder == "viterbi" else greedy
    pred = {}
    for doc, lp in zip(enc_docs, log_probs):
        real = doc["ends"] > doc["starts"]
        tags = fn(lp[real])
        pred[doc["id"]] = tags_to_spans(tags, doc["starts"][real], doc["ends"][real])
    return pred


def evaluate(model, enc_docs, cfg, device, pad_id, batch_size):
    lps = doc_log_probs(model, enc_docs, cfg, device, pad_id, batch_size)
    gold = {d["id"]: d["ents"] for d in enc_docs}
    results = {}
    for dec in ("viterbi", "greedy"):
        macro, per = strict_scores(decode_docs(enc_docs, lps, dec), gold)
        results[dec] = {"macro": macro, "per_label": per}
    return results


def boundary_report(pred, gold):
    """Splits the errors into: exact, right label but wrong boundary, overlapping with the wrong label, missed."""
    counts = {"exact": 0, "boundary": 0, "wrong_label": 0, "missed": 0, "spurious": 0}
    for doc_id, g in gold.items():
        p = list(pred.get(doc_id, []))
        pset = set(p)
        for s, e, lab in g:
            if (s, e, lab) in pset:
                counts["exact"] += 1
                continue
            over = [q for q in p if q[0] < e and q[1] > s]
            if any(q[2] == lab for q in over):
                counts["boundary"] += 1
            elif over:
                counts["wrong_label"] += 1
            else:
                counts["missed"] += 1
        for s, e, lab in p:
            if not any(gs < e and ge > s for gs, ge, _ in g):
                counts["spurious"] += 1
    return counts


def write_submission(pred, test_docs, out_dir, zip_name):
    os.makedirs(out_dir, exist_ok=True)
    payload = {}
    for doc in test_docs:
        n = len(doc["text"])
        spans = sorted(set(pred.get(doc["id"], [])))
        assert all(0 <= s < e <= n and lab in LABELS for s, e, lab in spans), f"bad span in {doc['id']}"
        payload[doc["id"]] = [[s, e, lab] for s, e, lab in spans]
    pred_path = os.path.join(out_dir, "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    zip_path = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]
    return pred_path, zip_path

Writing /kaggle/working/phase2_arch1/ner_core.py


In [8]:
%%writefile /kaggle/working/phase2_arch1/run_ner.py
"""
One Phase-2 experiment = one call:

    python run_ner.py --method lorma

Fine-tunes the Phase-1 model (or a distilled student of it) for legal NER, keeps the epoch with the best
dev strict macro-F1, writes the test predictions + submission zip and pushes everything to Hugging Face.
"""
import os
import sys
import json
import time
import random
import argparse

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
import backbone as BB
from ner_core import (load_docs, split_dev, encode_doc, window_items, collate, Tagger, configure_method,
                      count_params, trainable_state, build_optimizer, evaluate, doc_log_probs, decode_docs,
                      strict_scores, boundary_report, write_submission, N_TAGS, IGNORE)


def log(*a):
    print(*a, flush=True)


def build_tagger(method, keep_layers=None):
    model = Tagger(BB.build_backbone(keep_layers=keep_layers, bidirectional=C.BIDIRECTIONAL), C.HEAD_DROPOUT)
    note = configure_method(model, method, C)
    return model, note


def load_finetuned(method, device):
    """Rebuild a model trained earlier by this notebook (used as the KD teacher)."""
    path = os.path.join(C.OUT_DIR, method, f"ner_{method}.pt")
    if not os.path.exists(path):
        path = BB.fetch_from_hub(C.NER_REPO, f"{method}/ner_{method}.pt", os.path.join(C.OUT_DIR, "_hub"))
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"No fine-tuned '{method}' model found locally or on {C.NER_REPO}. "
                                f"Run the '{method}' experiment first - it is the distillation teacher.")
    blob = torch.load(path, map_location="cpu")
    model, _ = build_tagger(method, blob.get("keep_layers"))
    missing, unexpected = model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    assert not unexpected, unexpected
    return model.to(device).eval()


def class_weights(enc_docs, device):
    if C.CLASS_WEIGHTING != "sqrt_inv":
        return None
    counts = np.ones(N_TAGS)
    for d in enc_docs:
        t = d["tags"][d["tags"] != IGNORE]
        counts += np.bincount(t, minlength=N_TAGS)
    w = 1.0 / np.sqrt(counts / counts.sum())
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=device)


def push(files, message):
    if not os.environ.get("HF_TOKEN"):
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(C.NER_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
        for local, remote in files:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=C.NER_REPO,
                            token=os.environ["HF_TOKEN"], commit_message=message)
        log(f"pushed to {C.NER_REPO}: {message}")
    except Exception as e:
        log(f"HF push failed ({type(e).__name__}: {e}) - the files are still in {C.OUT_DIR}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["full", "partial", "lora", "lorma", "kd"])
    args = ap.parse_args()
    method = args.method

    random.seed(C.SEED)
    np.random.seed(C.SEED)
    torch.manual_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = os.path.join(C.OUT_DIR, method)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()

    # ---------------------------------------------------------------- data
    tok = BB.load_tokenizer()
    folds = [load_docs(C.FOLD_TEXT[i], C.FOLD_GOLD[i]) for i in range(2)]
    train_docs, dev_docs = split_dev(folds, C.DEV_DOCS_PER_FOLD, C.SEED)
    if C.TRAIN_DOC_LIMIT:                      # quick smoke test only
        train_docs, dev_docs = train_docs[:C.TRAIN_DOC_LIMIT], dev_docs[:max(1, C.TRAIN_DOC_LIMIT // 4)]
    test_docs = load_docs(C.TEST_TEXT)
    enc_train = [encode_doc(d, tok) for d in train_docs]
    enc_dev = [encode_doc(d, tok) for d in dev_docs]
    enc_test = [encode_doc(d, tok) for d in test_docs]
    log(f"[{method}] train docs {len(enc_train)} | dev docs {len(enc_dev)} | test docs {len(enc_test)} | "
        f"train tokens {sum(len(d['ids']) for d in enc_train):,}")

    # --------------------------------------------------------------- model
    keep = BB.student_layers() if method == "kd" else None
    model, note = build_tagger(method, keep)
    model.to(device)
    total, trainable = count_params(model)
    log(f"[{method}] {note} | params {total / 1e6:.1f}M, trainable {trainable / 1e6:.2f}M "
        f"({100 * trainable / total:.2f}%) | bidirectional attention: {C.BIDIRECTIONAL}")

    teacher = None
    if method == "kd":
        teacher = load_finetuned(C.KD_TEACHER, device)       # fp32 weights; autocast runs it in fp16 on the GPU
        t_total, _ = count_params(teacher)
        log(f"[kd] teacher = fine-tuned '{C.KD_TEACHER}' model ({t_total / 1e6:.1f}M) -> "
            f"student keeps layers {keep} ({total / 1e6:.1f}M)")

    # ------------------------------------------------------------ training
    items = window_items(enc_train, C.WINDOW, C.STRIDE)
    steps_per_epoch = math_ceil(len(items), C.MICRO_BATCH * C.GRAD_ACCUM)
    total_steps = steps_per_epoch * C.EPOCHS
    warmup = max(1, int(C.WARMUP_FRAC * total_steps))
    optimizer = build_optimizer(model, method, C)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    weights = class_weights(enc_train, device)
    rng = random.Random(C.SEED)

    best = {"f1": -1.0}
    stale, step = 0, 0
    history = []
    for epoch in range(1, C.EPOCHS + 1):
        model.train()
        rng.shuffle(items)
        running, seen = 0.0, 0
        for b in range(0, len(items), C.MICRO_BATCH):
            ids, tags, mask = collate(enc_train, items[b:b + C.MICRO_BATCH], tok.pad_id)
            ids, tags, mask = ids.to(device), tags.to(device), mask.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(ids, mask).float()
                loss = F.cross_entropy(logits.view(-1, N_TAGS), tags.view(-1), weight=weights, ignore_index=IGNORE)
                if teacher is not None:
                    # Hinton et al. 2015: match the teacher's softened tag distribution on every real token
                    with torch.no_grad():
                        t_logits = teacher(ids, mask).float()
                    keep_tok = tags.view(-1) != IGNORE
                    T = C.KD_TEMPERATURE
                    kl = F.kl_div(F.log_softmax(logits.view(-1, N_TAGS)[keep_tok] / T, -1),
                                  F.softmax(t_logits.view(-1, N_TAGS)[keep_tok] / T, -1),
                                  reduction="batchmean") * T * T
                    loss = C.KD_ALPHA * loss + (1 - C.KD_ALPHA) * kl
            scaler.scale(loss / C.GRAD_ACCUM).backward()
            running += loss.item()
            seen += 1

            if seen % C.GRAD_ACCUM == 0 or b + C.MICRO_BATCH >= len(items):
                # linear warm-up, then linear decay to zero
                if step < warmup:
                    frac = (step + 1) / warmup
                else:
                    frac = max(0.0, (total_steps - step) / max(1, total_steps - warmup))
                for g, lr in zip(optimizer.param_groups, base_lrs):
                    g["lr"] = lr * frac
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                step += 1

        res = evaluate(model, enc_dev, C, device, tok.pad_id, C.EVAL_BATCH)
        dec = max(res, key=lambda k: res[k]["macro"]["f1"])
        f1 = res[dec]["macro"]["f1"]
        history.append({"epoch": epoch, "train_loss": running / max(1, seen),
                        "dev_f1_viterbi": res["viterbi"]["macro"]["f1"], "dev_f1_greedy": res["greedy"]["macro"]["f1"]})
        log(f"[{method}] epoch {epoch}/{C.EPOCHS} | loss {running / max(1, seen):.4f} | dev strict macro-F1 "
            f"viterbi {res['viterbi']['macro']['f1']:.4f}  greedy {res['greedy']['macro']['f1']:.4f} | "
            f"{(time.time() - t0) / 60:.1f} min")
        if f1 > best["f1"]:
            best = {"f1": f1, "epoch": epoch, "decoder": dec, "dev": res[dec]}
            torch.save({"method": method, "keep_layers": keep, "weights": trainable_state(model),
                        "dev_f1": f1, "decoder": dec, "epoch": epoch}, os.path.join(out_dir, f"ner_{method}.pt"))
            stale = 0
        else:
            stale += 1
            if stale >= C.PATIENCE:
                log(f"[{method}] no dev improvement for {C.PATIENCE} epochs - stopping early")
                break

    # ------------------------------------------------- best model -> test
    blob = torch.load(os.path.join(out_dir, f"ner_{method}.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    t_inf = time.time()
    dev_lp = doc_log_probs(model, enc_dev, C, device, tok.pad_id, C.EVAL_BATCH)
    dev_pred = decode_docs(enc_dev, dev_lp, best["decoder"])
    dev_seconds = time.time() - t_inf
    errors = boundary_report(dev_pred, {d["id"]: d["ents"] for d in enc_dev})
    test_pred = decode_docs(enc_test, doc_log_probs(model, enc_test, C, device, tok.pad_id, C.EVAL_BATCH), best["decoder"])
    pred_path, zip_path = write_submission(test_pred, test_docs, out_dir, f"submission_{C.ARCH}_{method}.zip")

    summary = {
        "arch": C.ARCH, "method": method, "best_epoch": best["epoch"], "decoder": best["decoder"],
        "dev_macro": best["dev"]["macro"], "dev_per_label": best["dev"]["per_label"], "dev_errors": errors,
        "params_total": total, "params_trainable": trainable, "dev_inference_seconds": round(dev_seconds, 2),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else 0.0,
        "train_minutes": round((time.time() - t0) / 60, 1), "history": history,
        "test_entities": sum(len(v) for v in test_pred.values()),
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)
    log(f"[{method}] DONE | best dev strict macro-F1 {best['f1']:.4f} (epoch {best['epoch']}, {best['decoder']}) | "
        f"errors {errors} | {summary['test_entities']} test entities -> {zip_path}")

    push([(os.path.join(out_dir, f"ner_{method}.pt"), f"{method}/ner_{method}.pt"),
          (os.path.join(out_dir, "results.json"), f"{method}/results.json"),
          (zip_path, f"{method}/{os.path.basename(zip_path)}")], f"{C.ARCH} {method}: dev F1 {best['f1']:.4f}")


def math_ceil(a, b):
    return -(-a // b)


if __name__ == "__main__":
    main()

Writing /kaggle/working/phase2_arch1/run_ner.py


## 4. Phase-2 data

In [9]:
import importlib
import config as C
importlib.reload(C)

for name in ["Train/fold1-text.json", "Train/fold1-gold.json", "Train/fold2-text.json",
             "Train/fold2-gold.json", "Train/labels.txt", "Test/fold3-text.json"]:
    hf_hub_download(C.HF_DATASET_REPO, f"Phase_2_LNER/{name}", repo_type="dataset", local_dir=C.DATA_DIR, token=HF_TOKEN)

import ner_core as N
importlib.reload(N)
folds = [N.load_docs(C.FOLD_TEXT[i], C.FOLD_GOLD[i]) for i in range(2)]
test_docs = N.load_docs(C.TEST_TEXT)
with open(os.path.join(C.DATA_DIR, "Phase_2_LNER/Train/labels.txt")) as f:
    assert f.read().split() == N.LABELS, "label list changed - update LABELS in ner_core.py"

n_ents = sum(len(d["ents"]) for fold in folds for d in fold)
print(f"train docs {sum(map(len, folds))} | test docs {len(test_docs)} | gold entities {n_ents}")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

train docs 70 | test docs 35 | gold entities 8393


## 5. Tokenizer alignment and offset self-test
Strict scoring needs exact character offsets. Here every training document is encoded, the gold entities are turned into
BIO tags, and those tags are decoded straight back into spans. The resulting "oracle" F1 is the best score this
tokenization allows, and it doubles as a test of the whole offset pipeline.

In [10]:
import backbone as BB
importlib.reload(BB)

tok = BB.load_tokenizer()
all_docs = [d for fold in folds for d in fold]
enc = [N.encode_doc(d, tok) for d in all_docs]
oracle = {d["id"]: N.oracle_spans(d) for d in enc}
macro, per = N.strict_scores(oracle, {d["id"]: d["ents"] for d in enc})
print(f"oracle strict macro-F1 = {macro['f1']:.4f}  (the ceiling for this tokenizer)")
print("per label:", {lab: round(v["f1"], 3) for lab, v in per.items()})

n_tok = [len(d["ids"]) for d in enc]
print(f"tokens per document: mean {sum(n_tok) / len(n_tok):,.0f}, max {max(n_tok):,}")
if hasattr(tok, "unk_id"):
    unk = sum(int((d["ids"] == tok.unk_id).sum()) for d in enc) / sum(n_tok)
    print(f"unknown-word tokens: {100 * unk:.1f}%")
assert macro["f1"] > 0.95, "offset pipeline looks broken - do not submit"

custom_tokenizer.json:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

oracle strict macro-F1 = 0.9944  (the ceiling for this tokenizer)
per label: {'APP': 0.985, 'RESP': 0.995, 'A.COUNSEL': 1.0, 'R.COUNSEL': 1.0, 'JUDGE': 1.0, 'WIT': 1.0, 'AUTH': 0.991, 'COURT': 0.982, 'STAT': 0.984, 'PREC': 0.995, 'DATE': 1.0, 'CASENO': 1.0}
tokens per document: mean 6,420, max 23,264
unknown-word tokens: 2.6%


## 6. Load the Phase-1 weights once
This also caches them for the experiment processes.

In [11]:
probe = BB.build_backbone()
n = sum(p.numel() for p in probe.parameters())
ids = torch.tensor([enc[0]["ids"][:64].tolist()])
with torch.no_grad():
    h = probe(ids, torch.ones_like(ids, dtype=torch.bool))
print(f"backbone ready: {n / 1e6:.1f}M parameters, hidden states {tuple(h.shape)}")
del probe, h

config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

model_state_dict.pt: reconstructing file:   0%|          |  0.00B /  445MB            

model_state_dict.pt: downloading bytes:           |  0.00B            

Phase-1 arch 1 weights: /kaggle/working/phase2_arch1/phase1/d240357legalgpt2decoder/model_state_dict.pt (hf://D-240357/legal-gpt2-decoder)
backbone ready: 111.2M parameters, hidden states (1, 64, 768)


## 7. Run the experiments
Each experiment is its own process (`run_ner.py --method ...`); with two GPUs, two run at the same time.
`kd` starts once `full` (its teacher) has finished. Logs are prefixed with the GPU they run on.
Fine-tuned weights, results and submission zips are pushed to your NER repo on Hugging Face as they finish.

In [12]:
import subprocess, threading

pending = list(C.METHODS)
done, failed = set(), set()
lock = threading.Lock()


def next_job():
    with lock:
        if "kd" in pending and C.KD_TEACHER in failed:
            pending.remove("kd")
            print(f"skipping kd: its teacher '{C.KD_TEACHER}' failed")
        for m in pending:
            if m != "kd" or C.KD_TEACHER in done or C.KD_TEACHER not in C.METHODS:
                pending.remove(m)
                return m
        return "wait" if pending else None


def worker(gpu):
    while True:
        job = next_job()
        if job is None:
            return
        if job == "wait":
            time.sleep(15)
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        proc = subprocess.Popen([sys.executable, "run_ner.py", "--method", job], cwd=WORK_DIR, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(f"[gpu{gpu}] {line}", end="")
        with lock:
            (done if proc.wait() == 0 else failed).add(job)


t0 = time.time()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"\nfinished in {(time.time() - t0) / 60:.0f} min | ok: {sorted(done)} | failed: {sorted(failed)}")

[gpu1] [lorma] train docs 62 | dev docs 8 | test docs 35 | train tokens 407,617
[gpu0] [full] train docs 62 | dev docs 8 | test docs 35 | train tokens 407,617
[gpu1] Phase-1 arch 1 weights: /kaggle/working/phase2_arch1/phase1/d240357legalgpt2decoder/model_state_dict.pt (cache of hf://D-240357/legal-gpt2-decoder)
[gpu0] Phase-1 arch 1 weights: /kaggle/working/phase2_arch1/phase1/d240357legalgpt2decoder/model_state_dict.pt (cache of hf://D-240357/legal-gpt2-decoder)
[gpu0] [full] all weights trainable | params 111.2M, trainable 111.22M (100.00%) | bidirectional attention: True
[gpu1] [lorma] 24 linear layers wrapped with LORMA (r=8) | params 111.8M, trainable 0.61M (0.54%) | bidirectional attention: True
[gpu0] [full] epoch 1/10 | loss 0.7228 | dev strict macro-F1 viterbi 0.1351  greedy 0.0961 | 0.7 min
[gpu1] [lorma] epoch 1/10 | loss 0.9058 | dev strict macro-F1 viterbi 0.1287  greedy 0.0695 | 0.7 min
[gpu0] [full] epoch 2/10 | loss 0.2274 | dev strict macro-F1 viterbi 0.1581  greedy 0

## 8. Results
Dev scores (strict, the competition metric), cost of each method, and where the errors are.

In [13]:
import pandas as pd

rows, per_label, errors = [], {}, {}
for path in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "results.json"))):
    r = json.load(open(path))
    m = r["method"]
    rows.append({"method": m, "dev macro-F1": r["dev_macro"]["f1"], "dev P": r["dev_macro"]["p"],
                 "dev R": r["dev_macro"]["r"], "decoder": r["decoder"], "best epoch": r["best_epoch"],
                 "params (M)": r["params_total"] / 1e6, "trainable %": 100 * r["params_trainable"] / r["params_total"],
                 "train min": r["train_minutes"], "peak GPU GB": r["peak_gpu_gb"], "dev infer s": r["dev_inference_seconds"]})
    per_label[m] = {lab: v["f1"] for lab, v in r["dev_per_label"].items()}
    errors[m] = r["dev_errors"]

table = pd.DataFrame(rows).sort_values("dev macro-F1", ascending=False).round(4)
display(table)
display(pd.DataFrame(per_label).round(3))
display(pd.DataFrame(errors).T)

,method,dev macro-F1,dev P,dev R,decoder,best epoch,params (M),trainable %,train min,peak GPU GB,dev infer s
1,kd,0.2290,0.2522,0.2350,viterbi,9,68.6969,100.0000,4.9,2.67,1.00
3,lorma,0.2236,0.2713,0.2184,viterbi,8,111.8139,0.5447,5.9,2.12,1.72
2,lora,0.2052,0.2361,0.2037,viterbi,8,111.6665,0.4134,4.6,1.90,1.45
0,full,0.1982,0.2295,0.1954,viterbi,10,111.2241,100.0000,5.6,3.50,1.36
4,partial,0.1844,0.2080,0.1827,viterbi,9,111.2241,25.5091,3.5,1.42,1.43


,full,kd,lora,lorma,partial
APP,0.000,0.000,0.000,0.000,0.000
RESP,0.061,0.048,0.054,0.067,0.062
A.COUNSEL,0.000,0.000,0.000,0.071,0.000
R.COUNSEL,0.000,0.102,0.000,0.000,0.000
JUDGE,0.061,0.057,0.000,0.095,0.059
AUTH,0.125,0.202,0.242,0.248,0.152
COURT,0.809,0.821,0.826,0.819,0.845
STAT,0.298,0.284,0.264,0.263,0.287
PREC,0.057,0.209,0.157,0.148,0.031
DATE,0.671,0.698,0.653,0.685,0.563


,exact,boundary,wrong_label,missed,spurious
full,334,193,48,473,116
kd,377,184,63,424,140
lora,349,179,45,475,122
lorma,363,173,48,464,126
partial,308,212,33,495,147


## 9. Pick a submission
Every experiment already wrote `runs/<method>/submission_<arch>_<method>.zip` (a single `prediction.json` at the
root, every test document id present). The best one on dev is copied to `/kaggle/working/submission.zip` too.
Submitting several of them counts as different methods.

In [14]:
best = table.iloc[0]["method"]
src = glob.glob(os.path.join(C.OUT_DIR, best, "submission_*.zip"))[0]
shutil.copy(src, "/kaggle/working/submission.zip")
print(f"best on dev: {best} -> /kaggle/working/submission.zip")
for z in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "submission_*.zip"))):
    print("  ", z)

best on dev: kd -> /kaggle/working/submission.zip
   /kaggle/working/phase2_arch1/runs/full/submission_arch1_full.zip
   /kaggle/working/phase2_arch1/runs/kd/submission_arch1_kd.zip
   /kaggle/working/phase2_arch1/runs/lora/submission_arch1_lora.zip
   /kaggle/working/phase2_arch1/runs/lorma/submission_arch1_lorma.zip
   /kaggle/working/phase2_arch1/runs/partial/submission_arch1_partial.zip
